# Gold: lab_test_trends

Builds `gold.lab_test_trends` (one row per test type per calendar month) from silver `lab_results`, enriched with month and quarter names from the calendar dimension, with LAG-based month-over-month abnormal-rate trend, per the STTM `sttm_silver_to_gold.xlsx`. Load strategy: full rebuild (CREATE OR REPLACE TABLE ... AS SELECT).

Notes: `lab_results` is deduplicated to its latest record per `lab_result_id` before aggregation; the calendar dimension lives in the gold schema (`calendar`).

In [0]:
import re

# <catalog>/silver/gold placeholders from the STTM, mapped to this workspace's Unity Catalog names
dbutils.widgets.text("catalog", "man_catlog")
dbutils.widgets.text("silver_schema", "caresync_schema_silver")
dbutils.widgets.text("gold_schema", "caresync_schema_gold")

def _ident(name):
    v = dbutils.widgets.get(name).strip()
    if not re.fullmatch(r"[A-Za-z0-9_]+", v):
        raise ValueError(f"Invalid identifier for {name}: {v!r}")
    return v

catalog = _ident("catalog")
silver = f"{catalog}.{_ident('silver_schema')}"
gold = f"{catalog}.{_ident('gold_schema')}"
calendar_table = f"{gold}.calendar"   # calendar dimension was built in the gold schema
target_table = f"{gold}.lab_test_trends"
print(f"Source: {silver} | Calendar: {calendar_table} | Target: {target_table}")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {target_table}
COMMENT 'Gold: one row per test type per month with abnormal/critical rates and month-over-month trend (STTM silver->gold v1.0)'
AS
WITH lab_results AS (
  -- Silver holds duplicate rows per key; keep the latest record per lab result
  SELECT * FROM {silver}.lab_results
  QUALIFY ROW_NUMBER() OVER (PARTITION BY lab_result_id ORDER BY reported_at DESC, landing_ts DESC) = 1
),
monthly AS (
  SELECT
    test_name,
    YEAR(test_date)                       AS year,
    MONTH(test_date)                      AS month,
    COUNT(*)                              AS total_tests,
    COUNT_IF(result_status = 'Abnormal')  AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')  AS critical_test_count
  FROM lab_results
  GROUP BY test_name, YEAR(test_date), MONTH(test_date)
),
rates AS (
  SELECT *,
    CAST(ROUND(abnormal_test_count * 100.0 / total_tests, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(ROUND(critical_test_count * 100.0 / total_tests, 2) AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly
),
trend AS (
  SELECT *,
    LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month) AS prev_month_abnormal_rate_pct
  FROM rates
)
SELECT
  t.test_name,
  CAST(t.year AS INT)                                                    AS year,
  CAST(t.month AS INT)                                                   AS month,
  c.month_name,
  c.quarter_name,
  CAST(t.total_tests AS INT)                                             AS total_tests,
  CAST(t.abnormal_test_count AS INT)                                     AS abnormal_test_count,
  CAST(t.critical_test_count AS INT)                                     AS critical_test_count,
  t.abnormal_rate_pct,
  t.critical_rate_pct,
  CAST(t.prev_month_abnormal_rate_pct AS DECIMAL(5,2))                   AS prev_month_abnormal_rate_pct,
  CAST(t.abnormal_rate_pct - t.prev_month_abnormal_rate_pct AS DECIMAL(6,2)) AS abnormal_rate_change_pct
FROM trend t
LEFT JOIN {calendar_table} c
  ON c.date_key = CAST(date_format(make_date(t.year, t.month, 1), 'yyyyMMdd') AS INT)
""")
print(f"Created {target_table}")

In [0]:
chk = spark.sql(f"""
SELECT
  (SELECT COUNT(*) FROM {target_table})                                                AS gold_rows,
  (SELECT COUNT(*) FROM (SELECT DISTINCT test_name, year, month FROM {target_table}))  AS gold_distinct_keys,
  (SELECT SUM(total_tests) FROM {target_table})                                        AS gold_total_tests,
  (SELECT COUNT(DISTINCT lab_result_id) FROM {silver}.lab_results)                     AS silver_distinct_tests,
  (SELECT COUNT_IF(month_name IS NULL) FROM {target_table})                            AS rows_missing_calendar
""").first()
display(spark.createDataFrame([chk]))
assert chk.gold_rows == chk.gold_distinct_keys, "Grain violated: expected one row per test per month"
assert chk.gold_total_tests == chk.silver_distinct_tests, "Lab test totals do not reconcile with silver"
print("Validation passed")

In [0]:
display(spark.table(target_table).orderBy("test_name", "year", "month").limit(20))